# 06 - Sub-experiment 2: Prompt Engineering (Label Sensitivity)

This notebook tests how candidate-label phrasing changes zero-shot performance on the same shared 200-post sample.

In [1]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
import sys
sys.modules["tensorflow"] = None

### What this does and why

Zero-shot classification depends heavily on label wording. We evaluate three variants (simple, descriptive, clinical) and select the best-performing prompt for later experiments.

In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns

root = Path.cwd()
outputs_dir = root / "outputs"
if not outputs_dir.exists():
    outputs_dir = root.parent / "outputs"

sample_df = pd.read_csv(outputs_dir / "llm_sample.csv")
texts = sample_df["text_clean"].astype(str).tolist()
y_true = sample_df["risk_label"].astype(int).to_numpy()

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {
        "accuracy": float(acc),
        "f1_weighted": float(f1_w),
        "f1_macro": float(f1_m),
    }

try:
    classifier = pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli",
        device=-1,
    )
except Exception as e:
    raise RuntimeError(
        "Failed to load facebook/bart-large-mnli. Install transformers and torch. "
        f"Original error: {e}"
    )

prompt_variants = {
    "A": {
        "Labels Used": "simple",
        "labels": ["mental health risk", "suicidal risk"],
    },
    "B": {
        "Labels Used": "descriptive",
        "labels": ["general depression anxiety and loneliness", "suicidal ideation and self-harm"],
    },
    "C": {
        "Labels Used": "clinical",
        "labels": ["moderate psychological distress", "acute suicidal crisis requiring immediate intervention"],
    },
}

rows = []
for key, spec in prompt_variants.items():
    labels = spec["labels"]
    preds = []
    for t in tqdm(texts, desc=f"Prompt {key}"):
        out = classifier(t, candidate_labels=labels)
        winner = out["labels"][0]
        pred = 0 if winner == labels[0] else 1
        preds.append(pred)
    preds = np.array(preds, dtype=int)
    m = compute_metrics(y_true, preds)
    rows.append({
        "Prompt": key,
        "Labels Used": spec["Labels Used"],
        "label_0_text": labels[0],
        "label_1_text": labels[1],
        "Accuracy": m["accuracy"],
        "Weighted F1": m["f1_weighted"],
        "Macro F1": m["f1_macro"],
    })

prompt_df = pd.DataFrame(rows).sort_values("Weighted F1", ascending=False).reset_index(drop=True)
prompt_df.to_csv(outputs_dir / "prompt_engineering_results.csv", index=False)

best = prompt_df.iloc[0].to_dict()
(outputs_dir / "best_prompt.json").write_text(json.dumps(best, indent=2), encoding="utf-8")

plt.figure(figsize=(6, 4))
sns.barplot(data=prompt_df, x="Prompt", y="Weighted F1")
plt.ylim(0, 1)
plt.title("Prompt Sensitivity - Weighted F1")
plt.tight_layout()
plt.savefig(outputs_dir / "prompt_sensitivity_f1.png", dpi=200)
plt.close()

print("Best prompt:", best["Prompt"])
print("Labels:", [best["label_0_text"], best["label_1_text"]])
print("Weighted F1:", best["Weighted F1"])
prompt_df

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Prompt C: 100%|██████████| 200/200 [10:01<00:00,  3.01s/it]


Best prompt: B
Labels: ['general depression anxiety and loneliness', 'suicidal ideation and self-harm']
Weighted F1: 0.7033248081841432


,Prompt,Labels Used,label_0_text,label_1_text,Accuracy,Weighted F1,Macro F1
0,B,descriptive,general depression anxiety and loneliness,suicidal ideation and self-harm,0.710,0.703325,0.703325
1,A,simple,mental health risk,suicidal risk,0.685,0.657152,0.657152
2,C,clinical,moderate psychological distress,acute suicidal crisis requiring immediate inte...,0.535,0.413601,0.413601
